In [1]:

from complete_pipeline import CompletePipeline

pipeline = CompletePipeline(
    csv_path='../refactored_semantic/dbbe_full.csv',
    scratch_dir='/scratch/gent/vo/000/gvo00042/vsc48660',
    output_dir='./results',
    num_perms=[128],
    shingle_sizes=[3, 4],
    thresholds=[0.75, 0.8, 0.85],
    poem_thresholds=[0.2, 0.3, 0.4, 0.5],
    n_jobs=-1
)

pipeline.run_full_pipeline()


2025-11-27 10:38:29,737 - INFO - 
System detected: 32 cores, 376 GB RAM
2025-11-27 10:38:29,738 - INFO - → Workstation mode: Using 28 cores
2025-11-27 10:38:29,738 - INFO - → Large memory mode: 100000 chunk size
2025-11-27 10:38:29,741 - INFO - ======================================================================
2025-11-27 10:38:29,741 - INFO - COMPLETE PRODUCTION PIPELINE
2025-11-27 10:38:29,741 - INFO - ======================================================================
2025-11-27 10:38:29,741 - INFO - Scratch: /scratch/gent/vo/000/gvo00042/vsc48660/complete_clustering
2025-11-27 10:38:29,742 - INFO - Verse grid search: 1×2×3 = 6 configs
2025-11-27 10:38:29,742 - INFO - Poem grid search: 4 thresholds
2025-11-27 10:38:29,742 - INFO - 
Step 1: Loading and validating data
2025-11-27 10:38:30,055 - WARNING - Removing 7 empty verses
2025-11-27 10:38:30,063 - WARNING - Removing 12,739 verses with NaN ground truth
2025-11-27 10:38:30,069 - INFO - Loaded 41,424 verses from 10,821 poems


## Print sample results

In [3]:
import pandas as pd
import numpy as np
df=pd.read_csv('results/verse_clusters.csv')
df=df.dropna()
def show_example_verse_clusters(df: pd.DataFrame, n_clusters: int = 3):
    clusters = (
        df['predicted_verse_cluster']
        .dropna()
        .astype(int)
        .unique()
    )
    cluster_sizes = (
    df.groupby("predicted_verse_cluster")
      .size()
      .rename("count")
    )

    clusters_with_multiple = cluster_sizes[cluster_sizes > 1].index.tolist()
    example_clusters = np.random.choice(clusters_with_multiple, size=min(n_clusters, len(clusters)), replace=False)
    
    for cid in example_clusters:
        print(f"Cluster {cid}")
        
        subset = df[df['predicted_verse_cluster'] == cid]
        
        for _, row in subset.iterrows():
            # print(f"Poem {row['idoriginal_poem']:>8} | Verse group {row['idgroup']:>3} | Verse: {row['verse']}")
            print(f"Verse: {row['verse']}")
        
        print("\n")
show_example_verse_clusters(df, n_clusters=10)

Cluster 38720
Verse: ἔστι μὲν οὖν ὁ πυρρίχιος μὲν λόγος·
Verse: ἔστωσαν οὖν σοι, | πυρρίχιος μὲν λόγος·


Cluster 23598
Verse: Ψαλμῶν τὸ τεῦχο(ς), ἐσφραγισμένη χάρις·
Verse: <ψ>αλμῶν τὸ τεῦχος ἐσφραγισμένοις χάρις,


Cluster 4359
Verse: τρισσουμένων ἦν ὁ δρόμο(ς) σὺν ἑξάδι:-
Verse: τρισσουμένων ἦν ὁ δρόμο(ς) σὺν ἑξάδι:-


Cluster 16464
Verse: οὕτω γε καὶ νῦν ἡ φάλαγξ τῶν δαιμόνων
Verse: οὕτως γε καὶ νῦν ἡ φάλαγξ τῶν δαιμόνων,
Verse: οὕτως γε καὶ νῦν ἡ φάλαγξ τῶν δαιμόνων,
Verse: οὕτως γε καὶ νῦν ἡ φάλαγξ τῶν δαιμόνων


Cluster 18707
Verse: ἀντὶ δίκας πολίης, ἱερὸν μετἐκείαθε παῦλον :-
Verse: ἀντὶ δίκας πολίης ἱερὸν μετ᾿ἐκείαθε παῦλον +


Cluster 39957
Verse: ἀσκαρδαμύκτ(ως) τ(ὸν) νοητ(ὸν) φωσφόρον·,
Verse: ἀσκαρ|δαμύκτ(ως) τ(ὸν) νοητ(ὸν) | φωσφόρον·


Cluster 12069
Verse: οὐ μὲν ἐγὼ τοιοῦτον ἴδόν ποτε πέπλον ἀθήνης
Verse: οὐ μὲν ἐγὼ τοιοῦτον ἴδον ποτὲ πέπλον Ἀθήνης


Cluster 9552
Verse: ἀρὰς φρικώδεις λήψεται τῶν ἁγί(ων)
Verse: ἀρὰς φρικώδεις λήψεται τῶν ἁγί(ων)


Cluster 26643
Verse: 

In [4]:
import pandas as pd

# Load poems and concatenated metadata
poems_df = pd.read_csv("results/poem_clusters.csv")
poems_df["idoriginal_poem"] = poems_df["idoriginal_poem"].astype(str)

# Find clusters with more than 1 poem
cluster_sizes = poems_df.groupby("predicted_poem_cluster").size()
multi_poem_clusters = cluster_sizes[cluster_sizes > 1].index.tolist()
print(f"Found {len(multi_poem_clusters)} multi-poem clusters.")

# Take first 3 clusters for inspection
clusters_to_show = multi_poem_clusters[:3]
poems_to_show = poems_df[poems_df["predicted_poem_cluster"].isin(clusters_to_show)]

# Create a lookup dict: cluster -> list of poem IDs
cluster_to_poems = (
    poems_to_show.groupby("predicted_poem_cluster")["idoriginal_poem"]
    .apply(list)
    .to_dict()
)

# Now, read concatenated.csv **in chunks** to handle large files
verse_dict = {}  # poem_id -> list of verses
chunksize = 100_000

for chunk in pd.read_csv("../concatenated.csv", chunksize=chunksize):
    chunk["idoriginal_poem"] = chunk["idoriginal_poem"].astype(str)
    chunk = chunk[chunk["idoriginal_poem"].isin(poems_to_show["idoriginal_poem"])]
    
    # Group by poem and sort by order
    for poem_id, group in chunk.groupby("idoriginal_poem"):
        verses = group.sort_values("order")["verse"].tolist()
        # Convert non-string verses to str, skip NaNs
        verses = [str(v) for v in verses if pd.notna(v)]
        if poem_id in verse_dict:
            verse_dict[poem_id].extend(verses)
        else:
            verse_dict[poem_id] = verses

# Reconstruct poems for selected clusters
for cl in clusters_to_show:
    print("\n" + "="*80)
    print(f"CLUSTER {cl}")
    print("="*80)
    
    for poem_id in cluster_to_poems[cl]:
        poem_text = "\n".join(verse_dict.get(poem_id, []))
        print(f"\n--- Poem {poem_id} ---\n")
        print(poem_text)
        print("\n" + "-"*40)


Found 832 multi-poem clusters.

CLUSTER 2

--- Poem 16899 ---

πέτρου μυηθεὶς τοῖς ἀπορρήτοις λόγοις:
τὴν τοῦ θ(εο)ῦ κένωσιν εἰς βροτῶν φύσιν:
ἐν ἧ τὸ διπλοῦν ὢν θεάνθρωπος φέρει:
ταύτην καθεξῆς συντίθησι πανσόφως:
ὁ δευτερεύων μᾶρκος ἐν θεογράφοις
ὑ(*)πατείας τῶν δεσπ[οτ]ῶν ἡμῶν Κ̣ω̣ν̣σ̣τ̣[αντίνου]
καὶ Λικιννίου σεβ[α]στῶν τ̣[ὸ]δ̣
Αὐρηλίῳ Διοσκουρίδου(*) τῷ καὶ Ἰουλιανυ(*)γ̣υ̣μ̣[νασ]ι̣α̣[ρ]χ̣ο͂ν-
τι(*) πρυτανεύσαντι βου(λευτῇ) τῆς λαμ(πρᾶς) καὶ λα[μ(προτάτης)]Ὀξυρ(υγχειτῶν) πό[λ]εως
παρὰ Αὐρηλίου Λεονίδου Θέωνος ἀπὸ τῆς αὐτῆς πόλεως.
ἑκουσίως ἐπιδέχομε(*) μισ[θ]ώσασθαι πρὸς μ[ό]ν[ο]ν τὸ ἐ-
νεστὸς ι καὶ η ἔτος ἀπὸ τῶν ὑ(*)παρχόντω[ν]ἐ̣ν̣ περι-
χώματι Πέκτυ ἐδάφους Καράβου λεγομέ[νο]υ ἀπὸ νω-
τίνων(*) ἀρουρῶν δέκα  ἐκ τοῦ ἀπὸ λιβὸς ἐπὶ   ̣[  ̣]  ̣οσ̣ι̣ μέρεσ-
ιν τὰς ἀπὸ ἀναπαύσεως ἀρούρας δύο ἥμισοι(*)[ἐ]κ γεομετρί-
ας(*) ἢ ὅσας̣(*)ἒ̣ν(*) ὧσι σει(*) σπορὰν νιλοκαλάμης(*) φόρου ἑκά-
στης ἀρούρης ἀνὰ ἀργυρίου ταλάντων τεσάρω[ν](*) ἀκίνδυ-
να πάν[τα] παντὸς κινδ[ύ]νου τῶν τῆς γῆ[ς] δημοσί-